# Employee Data Cleaning Project

Cleaning a messy, synthetic HR dataset (1,020 rows, 12 columns) sourced from Kaggle. This notebook documents every issue found and the reasoning behind each fix.

Unlike a dataset full of obvious, surface-level mess, this one looks deceptively clean at first glance. The real risk here is silent corruption: issues that do not throw an error and do not show up unless you go looking for them. Three findings in particular required questioning a result that looked fine on the surface: a phone-number column that turned out to be a hash artifact (100% negative values), a validation check that silently miscounted missing values as decimals, and a date conversion that reported success without guaranteeing every date was interpreted in the correct order. Each is called out below at the step where it was found.

## Step 1: Load the data and take a first look

In [1]:
import pandas as pd
import numpy as np

# Phone is loaded as string from the start: purely numeric columns are
# auto-detected as int/float by pandas, which can silently corrupt
# identifier-like fields (leading characters lost, sign artifacts, etc.)
df = pd.read_csv('Messy_Employee_dataset.csv', dtype={'Phone': str})

df.shape

(1020, 12)

In [2]:
df.dtypes

Employee_ID           object
First_Name            object
Last_Name             object
Age                  float64
Department_Region     object
Status                object
Join_Date             object
Salary               float64
Email                 object
Phone                 object
Performance_Score     object
Remote_Work             bool
dtype: object

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   object 
 1   First_Name         1020 non-null   object 
 2   Last_Name          1020 non-null   object 
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   object 
 5   Status             1020 non-null   object 
 6   Join_Date          1020 non-null   object 
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   object 
 9   Phone              1020 non-null   object 
 10  Performance_Score  1020 non-null   object 
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), object(9)
memory usage: 88.8+ KB


**Finding:** `Age` has 211 missing values (809/1020) and `Salary` has 24 missing (996/1020). All other columns are fully populated. `Phone` is correctly loaded as `object` thanks to the `dtype` parameter above — loading it without that parameter causes pandas to interpret it as `int64`.

## Step 2: Preview the data

In [4]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [5]:
df.sample(5, random_state=1)

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
267,EMP1267,Heidi,Garcia,NaN,HR-Florida,Active,1/24/2022,73186.77,heidi.garcia@example.com,-8621102257,Good,False
142,EMP1142,Eva,Brown,25.0,Sales-Nevada,Pending,4/10/2020,113743.19,eva.brown@example.com,-6346692547,Good,True
839,EMP1839,Charlie,Jones,35.0,DevOps-New York,Inactive,1/5/2021,105956.04,charlie.jones@example.com,-7123166006,Excellent,False
175,EMP1175,Eva,Miller,25.0,Admin-California,Pending,4/10/2024,71628.93,eva.miller@example.com,-8155839385,Excellent,False
481,EMP1481,Eva,Miller,35.0,DevOps-Texas,Active,3/5/2022,88945.49,eva.miller@example.com,-1935889535,Good,False


In [6]:
df.describe()

,Age,Salary
count,809.000000,996.000000
mean,32.484549,85155.056396
std,5.656860,19873.727918
min,25.000000,50047.320000
25%,25.000000,68392.487500
50%,30.000000,85547.870000
75%,40.000000,100974.027500
max,40.000000,119971.650000


**Findings:**
- `Department_Region` is a compound column (e.g. `"DevOps-California"`) that needs to be split.
- `Join_Date` is stored as text (e.g. `4/2/2021`) and needs conversion to a real date type.
- `Age` ranges from 25 to 40 — a narrow range, consistent with this being a synthetic dataset.

## Step 3-4: Missing values and duplicates

In [7]:
df.isnull().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

**Finding:** No duplicate rows. Missing values are limited to `Age` (211) and `Salary` (24) — both numeric columns that can reasonably be left as `NaN` for now, since neither can be reliably imputed from other columns.

## Step 5: Standardize column names

In [9]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Index(['employee_id', 'first_name', 'last_name', 'age', 'department_region',
       'status', 'join_date', 'salary', 'email', 'phone', 'performance_score',
       'remote_work'],
      dtype='object')

## Step 6: Fix data types

In [10]:
df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')
df['join_date'].isnull().sum()   # 0 -> no format conflicts detected

np.int64(0)

In [11]:
df.dtypes

employee_id                  object
first_name                   object
last_name                    object
age                         float64
department_region            object
status                       object
join_date            datetime64[ns]
salary                      float64
email                        object
phone                        object
performance_score            object
remote_work                    bool
dtype: object

**Note:** Zero conversion failures doesn't guarantee the month/day order is correct for every row — `pd.to_datetime` silently accepts any date where the day value is 12 or below, regardless of whether it was actually meant as day-first or month-first. Absent documentation stating the source format, this residual ambiguity is disclosed here rather than hidden.

## Step 7: Split compound column and check categorical consistency

In [12]:
df[['department', 'region']] = df['department_region'].str.split('-', n=1, expand=True)
df = df.drop(columns=['department_region'])
df[['department', 'region']].head()

,department,region
0,DevOps,California
1,Finance,Texas
2,Admin,Nevada
3,Admin,Nevada
4,Cloud Tech,Florida


In [13]:
df['department'].value_counts()

department
DevOps        189
Sales         178
HR            171
Finance       170
Admin         166
Cloud Tech    146
Name: count, dtype: int64

In [14]:
df['region'].value_counts()

region
California    187
Florida       185
Nevada        169
Illinois      165
New York      161
Texas         153
Name: count, dtype: int64

In [15]:
df['status'].value_counts()

status
Pending     356
Active      352
Inactive    312
Name: count, dtype: int64

In [16]:
df['performance_score'].value_counts()

performance_score
Good         270
Average      267
Excellent    267
Poor         216
Name: count, dtype: int64

**Finding:** All four categorical columns are already clean — no casing or spelling inconsistencies. No changes needed here.

## Step 8: Outlier and validity checks

In [17]:
# NaN-safe check: NaN % 1 != 0 evaluates to True in pandas, which would
# otherwise miscount missing values as decimal ages.
(df['age'].dropna() % 1 != 0).sum()

np.int64(0)

In [18]:
df[['age', 'salary']].describe()

,age,salary
count,809.000000,996.000000
mean,32.484549,85155.056396
std,5.656860,19873.727918
min,25.000000,50047.320000
25%,25.000000,68392.487500
50%,30.000000,85547.870000
75%,40.000000,100974.027500
max,40.000000,119971.650000


In [19]:
# sort_values() always places NaN last regardless of sort direction,
# so dropna() first to see genuine minimum/maximum values.
df['salary'].dropna().sort_values().head(5)

394    50047.32
858    50060.73
782    50110.66
549    50153.79
390    50173.12
Name: salary, dtype: float64

In [20]:
df['salary'].dropna().sort_values().tail(5)

444    119586.11
96     119764.20
439    119801.30
712    119890.35
243    119971.65
Name: salary, dtype: float64

**Finding:** `age` has no decimal values once missing entries are excluded. `salary` ranges from ~$50K to ~$120K with no outliers.

**Finding — `phone`:** Every single value (1020/1020) starts with a `-` sign, and value lengths are inconsistent (10, 9, 8, or 7 digits once the sign is removed). A 100% occurrence rate rules out random corruption — this points to a systematic artifact of how the synthetic dataset was generated (most likely a hash-based ID that happened to double as the phone field), not real phone numbers. Values with fewer than 10 digits cannot represent a valid phone number and are treated as invalid.

In [21]:
df['phone'] = df['phone'].str.lstrip('-')
df.loc[df['phone'].str.len() < 10, 'phone'] = np.nan
df['phone'].str.len().value_counts(dropna=False)

phone
10.0    928
NaN      92
Name: count, dtype: int64

## Step 9: Save the cleaned data and document before/after

In [22]:
# Reload the original file separately so the comparison reflects the
# true starting point, independent of any in-memory state above.
df_raw = pd.read_csv('Messy_Employee_dataset.csv')

comparison = pd.DataFrame({
    'missing_before': df_raw.isnull().sum(),
})
comparison.index = comparison.index.str.lower().str.replace(' ', '_')
comparison = comparison.reindex(df.columns.union(['department_region'], sort=False))

missing_after = df.isnull().sum()
# department_region no longer exists after cleaning; department/region replace it
comparison['missing_after'] = comparison.index.map(missing_after).fillna('column split into department/region')
comparison

,missing_before,missing_after
employee_id,0.0,0.0
first_name,0.0,0.0
last_name,0.0,0.0
age,211.0,211.0
status,0.0,0.0
join_date,0.0,0.0
salary,24.0,24.0
email,0.0,0.0
phone,0.0,92.0
performance_score,0.0,0.0


In [23]:
df.to_csv('employee_data_cleaned.csv', index=False)
print(f'Saved {len(df)} rows, {len(df.columns)} columns to employee_data_cleaned.csv')

Saved 1020 rows, 13 columns to employee_data_cleaned.csv


In [24]:
df.head()

,employee_id,first_name,last_name,age,status,join_date,salary,email,phone,performance_score,remote_work,department,region
0,EMP1000,Bob,Davis,25.0,Active,2021-04-02,59767.65,bob.davis@example.com,1651623197,Average,True,DevOps,California
1,EMP1001,Bob,Brown,NaN,Active,2020-07-10,65304.66,bob.brown@example.com,1898471390,Excellent,True,Finance,Texas
2,EMP1002,Alice,Jones,NaN,Pending,2023-12-07,88145.90,alice.jones@example.com,5596363211,Good,True,Admin,Nevada
3,EMP1003,Eva,Davis,25.0,Inactive,2021-11-27,69450.99,eva.davis@example.com,3476490784,Good,True,Admin,Nevada
4,EMP1004,Frank,Williams,25.0,Active,2022-01-05,109324.61,frank.williams@example.com,1586734256,Poor,False,Cloud Tech,Florida


## Summary

| Issue | Column(s) | Resolution |
|---|---|---|
| Identifier misread as numeric | `phone` | Reloaded with `dtype=str` |
| Missing values | `age`, `salary` | Left as `NaN` (not reliably imputable) |
| Compound column | `department_region` | Split into `department` and `region` |
| Inconsistent column names | all | Converted to `snake_case` |
| Text date | `join_date` | Converted to `datetime64[ns]` |
| Invalid values | `phone` | Sign stripped; entries under 10 digits set to `NaN` |
| Duplicates | — | None found |
| Categorical inconsistency | `status`, `performance_score`, `department`, `region` | None found — already clean |
